# Evo-1 LIBERO FP16 Resumable Reference — A100

No quantization. This keeps the Evo-1 server/client split, but avoids one giant fragile client run.

The client runs one suite/task/episode per subprocess and saves every episode log plus a `.done.json` marker to Drive. If Colab stops, rerun the notebook and completed runs are skipped.


In [6]:
# 0. GPU check
import torch, os, subprocess, sys, textwrap, json, re, time
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU")


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


In [7]:
# 1. Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

BASE    = "/content/drive/MyDrive/cat_baseline"
REPO    = f"{BASE}/Evo-1"
EVO     = f"{REPO}/Evo_1"
LIBERO_EVAL = f"{REPO}/LIBERO_evaluation"
CKPT_DIR    = f"{BASE}/checkpoints/Evo1_LIBERO"
RESULTS     = f"{BASE}/results"

import os
for p in [BASE, CKPT_DIR, RESULTS]:
    os.makedirs(p, exist_ok=True)
print(BASE, REPO, EVO, LIBERO_EVAL, CKPT_DIR, RESULTS, sep="
")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Evo-1
/content/drive/MyDrive/Evo-1/Evo_1
/content/drive/MyDrive/Evo-1/LIBERO_evaluation
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO
/content/drive/MyDrive/Evo-1-results/fp16_resumable


In [8]:
# 2. Clone fresh Evo-1 into cat_baseline — always reset to origin/main
import os
os.makedirs(BASE, exist_ok=True)
%cd "$BASE"
if not os.path.exists(REPO):
    !git clone https://github.com/MINT-SJTU/Evo-1.git Evo-1
else:
    print("Repo exists, resetting to origin/main")

%cd "$REPO"
!git fetch origin
!git reset --hard origin/main
!git clean -fd
!git status --short
print("Repo is clean — no quantization patches applied")


/content/drive/MyDrive
Cloning into 'Evo-1'...
remote: Enumerating objects: 1226, done.
remote: Counting objects: 100% (563/563), done.
remote: Compressing objects: 100% (415/415), done.
remote: Total 1226 (delta 164), reused 524 (delta 140), pack-reused 663 (from 1)
Receiving objects: 100% (1226/1226), 7.53 MiB | 18.58 MiB/s, done.
Resolving deltas: 100% (306/306), done.
Updating files: 100% (417/417), done.
/content/drive/MyDrive/Evo-1
 M .gitignore
 M Evo_1/dataset/config.yaml
 M Evo_1/ds_config.json
 M Evo_1/model/action_head/flow_matching.py
 M Evo_1/scripts/Evo1_server.py
 M MetaWorld_evaluation/mt50_evo1_client_prompt.py
 M MetaWorld_evaluation/tasks.jsonl
 M so100_evo1/lerobot-main/benchmarks/video/capture_camera_feed.py
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/dataset/config.yaml
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/ds_config.json
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/model/action_head/flow_matching.py
 M so100_evo1/lerobot-main/src/l

In [9]:
# 3. Install micromamba and create envs
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

if not os.path.exists(MAMBA):
    !wget -qO /tmp/micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
    !mkdir -p /content/micromamba
    !tar -xjf /tmp/micromamba.tar.bz2 -C /content/micromamba bin/micromamba

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n Evo1 python=3.10 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n libero python=3.8.13 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} env list


[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (1.6 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.6 sec)

Transaction

  Prefix: /content/micromamba-root/envs/Evo1

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel          Size
─────────────────────────────────────────────────────────

In [11]:
# 4. Install Evo-1 server deps
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/cat_baseline/Evo-1/Evo_1"

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -U pip setuptools wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -r "{EVO}/requirements.txt"
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install --force-reinstall "huggingface-hub==0.36.2"

import subprocess, os

check_code = '''
import transformers, huggingface_hub, torch
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
'''

subprocess.run(
    [MAMBA, "run", "-n", "Evo1", "python", "-c", check_code],
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    check=True,
)

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached requests-2.34.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached charset_normalizer-3.4.7-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached idna-3.15-py3-none-any.whl.metadata (7.7 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-n

CompletedProcess(args=['/content/micromamba/bin/micromamba', 'run', '-n', 'Evo1', 'python', '-c', '\nimport transformers, huggingface_hub, torch\nprint("transformers:", transformers.__version__)\nprint("huggingface_hub:", huggingface_hub.__version__)\nprint("torch:", torch.__version__)\nprint("CUDA:", torch.cuda.is_available())\n'], returncode=0)

In [12]:
# 5. A100: try flash-attn, but do not force-disable it
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/cat_baseline/Evo-1/Evo_1"
!cd "{EVO}" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 bash -lc 'MAX_JOBS=4 pip install -v flash-attn --no-build-isolation' || echo "flash-attn install failed/skipped"


Using pip 26.1.1 from /content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/pip (python 3.10)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 94.3 MB/s  0:00:00
  Running command Preparing metadata (pyproject.toml)
  /content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
  !!

          ********************************************************************************
          Please consider removing the following classifiers in favor of a SPDX license expression:

          License :: OSI Approved :: BSD License

          See https://p

In [13]:
# 6. Install LIBERO env
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
LIBERO_EVAL = "/content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation"

!cd "{LIBERO_EVAL}" && test -d LIBERO || git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -U "pip<25.1" "setuptools<76" wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install "numpy<1.24" "protobuf<4"
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -r requirements.txt
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -e .
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install websockets==13.1 huggingface_hub imageio imageio-ffmpeg opencv-python


Cloning into 'LIBERO'...
remote: Enumerating objects: 1788, done.
remote: Counting objects: 100% (455/455), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 1788 (delta 290), reused 290 (delta 290), pack-reused 1333 (from 1)
Receiving objects: 100% (1788/1788), 315.98 MiB | 19.13 MiB/s, done.
Resolving deltas: 100% (755/755), done.
Updating files: 100% (1116/1116), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.3.0
    Uninstalling setuptools-75.3.0:
      Successfully uninstalled setuptools-75.3.0
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 17.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0

In [14]:
# 7. Download checkpoint
from huggingface_hub import snapshot_download
import os
CKPT_DIR = "/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO"
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(repo_id="MINT-SJTU/Evo1_LIBERO", local_dir=CKPT_DIR, local_dir_use_symlinks=False)
!find "{CKPT_DIR}" -maxdepth 2 -type f | sort | head -50


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/checkpoint.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/config.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/mp_rank_00_model_states.pt
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/norm_stats.json


In [16]:
# 8. Patch server only: checkpoint path, port 9010, websocket no-timeout
from pathlib import Path
import re

REPO = "/content/drive/MyDrive/cat_baseline/Evo-1"
EVO = f"{REPO}/Evo_1"
CKPT_DIR = "/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO"

server_path = Path(f"{EVO}/scripts/Evo1_server.py")
txt = server_path.read_text()

txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{CKPT_DIR}"',
    txt,
    count=1,
)

txt = txt.replace("9000", "9010")

if "ping_interval=None" not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
    txt = txt.replace(
        "websockets.serve(handler, '0.0.0.0', PORT)",
        "websockets.serve(handler, '0.0.0.0', PORT, ping_interval=None, ping_timeout=None, close_timeout=30)",
    )

server_path.write_text(txt)

!grep -n "ckpt_dir\|9010\|9000\|websockets.serve\|ping_interval" "{server_path}" | tail -40

63:def load_model_and_normalizer(ckpt_dir):
64:    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
65:    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
72:    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")
149:    ckpt_dir = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
150:    #Example: ckpt_dir = "/home/dell/checkpoints/Evo1/Evo1_MetaWorld/"
152:    port = 9010
155:    model, normalizer = load_model_and_normalizer(ckpt_dir)
159:        async with websockets.serve(


In [18]:
%%bash
# 9. LIBERO config

mkdir -p ~/.libero
mkdir -p /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets

cat > ~/.libero/config.yaml <<'EOF'
benchmark_root: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets
EOF

cat ~/.libero/config.yaml

benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets


In [57]:
# 10. Create runtime single-episode LIBERO client copy — fixed clean version
from pathlib import Path
import re

LIBERO_EVAL = "/content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation"
src = Path(f"{LIBERO_EVAL}/libero_client_4tasks.py")
dst = Path(f"{LIBERO_EVAL}/libero_client_single_episode_runtime.py")

original = src.read_text()

prefix = """
import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------
"""

txt = prefix + "\n" + original

txt = re.sub(
    r"SERVER_URL\s*=\s*['\"].*?['\"]",
    'SERVER_URL = "ws://127.0.0.1:9010"',
    txt,
    count=1,
)

txt = re.sub(r"horizon\s*=\s*\d+", "horizon = 14", txt, count=1)
txt = re.sub(r"max_steps\s*=\s*\[[^\]]+\]", "max_steps = [SINGLE_MAX_STEPS]", txt, count=1)
txt = re.sub(r"task_suites\s*=\s*\[[^\]]+\]", "task_suites = [SINGLE_SUITE]", txt, count=1)
txt = re.sub(r"num_episodes\s*=\s*\d+", "num_episodes = 1", txt, count=1)
txt = re.sub(r"ckpt_name\s*=\s*f?['\"].*?['\"]", "ckpt_name = SINGLE_CKPT_NAME", txt, count=1)

txt = txt.replace("for task_id in range(num_tasks_in_suite):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 1)):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 10)):", "for task_id in [SINGLE_TASK_ID]:")

txt = txt.replace("initial_states[episode_id]", "initial_states[SINGLE_EP_INDEX]")
txt = txt.replace("init_states[episode_id]", "init_states[SINGLE_EP_INDEX]")
txt = txt.replace("for episode_id in range(num_episodes):", "for episode_id in [0]:")

txt = txt.replace(
    "async with websockets.connect(SERVER_URL) as ws:",
    "async with websockets.connect(SERVER_URL, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:",
)

dst.write_text(txt)

print("Wrote:", dst)
print("\nTop of generated runtime client:")
print("\n".join(dst.read_text().splitlines()[:20]))

# Remove failed marker from the previous crashed run, so Cell 15 reruns it.
RESULTS = Path("/content/drive/MyDrive/cat_baseline/results")
for name in [
    "fp16_baseline_debug_libero_spatial_task0_ep0.log",
    "fp16_baseline_debug_libero_spatial_task0_ep0.done.json",
]:
    p = RESULTS / name
    if p.exists():
        p.unlink()
        print("deleted failed previous file:", p)

Wrote: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/libero_client_single_episode_runtime.py

Top of generated runtime client:

import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------

import asyncio
import websockets
import numpy as np
import json
import pathlib
import os
deleted failed previous file: /content/drive/MyDrive/Evo-1-results/fp16_resumable/fp16_resumable_debug_libero_spatial_task0_ep0.log
deleted failed previous file: /content/drive/MyDrive/Evo-1-results/fp16_resumable/fp16_resumable_debug_libero_s

In [43]:
# Fast local checkpoint copy + server restart
import subprocess, os, re, time
from pathlib import Path

DRIVE_CKPT = "/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO"
LOCAL_CKPT = "/content/Evo1_LIBERO"
EVO = "/content/drive/MyDrive/cat_baseline/Evo-1/Evo_1"
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
SERVER_LOG = "/content/evo1_fp16_server.log"

# Copy checkpoint from Drive to local Colab disk
subprocess.run(["mkdir", "-p", LOCAL_CKPT], check=True)
subprocess.run(["rsync", "-ah", "--info=progress2", DRIVE_CKPT + "/", LOCAL_CKPT + "/"], check=True)

# Patch server to use local checkpoint path
server_path = Path(f"{EVO}/scripts/Evo1_server.py")
txt = server_path.read_text()
txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{LOCAL_CKPT}"',
    txt,
    count=1,
)
txt = txt.replace("9000", "9010")
if "ping_interval=None" not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
server_path.write_text(txt)

# Restart server
subprocess.run(["pkill", "-f", "Evo1_server.py"], check=False)

server_proc = subprocess.Popen(
    [MAMBA, "run", "-n", "Evo1", "python", "-u", "scripts/Evo1_server.py"],
    cwd=EVO,
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
)

print("server pid:", server_proc.pid)
time.sleep(20)
print(Path(SERVER_LOG).read_text(errors="ignore").splitlines()[-80:])

server pid: 18760
['Loading EVO_1 model...', '/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.', '  warnings.warn(', 'Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.', '/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.', '  warnings.warn(', 'num_inference_timesteps 32', "/content/drive/MyDrive/Evo-1/Evo_1/scripts/Evo1_server.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the def

In [48]:
!ps -ef | grep Evo1_server.py | grep -v grep || echo "SERVER DEAD"
!ss -ltnp | grep 9010 || echo "PORT 9010 NOT OPEN"
!nvidia-smi --query-gpu=name,memory.used,memory.free,utilization.gpu --format=csv,noheader
!tail -n 120 /content/evo1_fp16_server.log || true

root       18760    1751  0 11:09 ?        00:00:00 /content/micromamba/bin/micromamba run -n Evo1 python -u scripts/Evo1_server.py
root       18762   18760  9 11:09 ?        00:00:14 python -u scripts/Evo1_server.py
LISTEN 0      100          0.0.0.0:9010       0.0.0.0:*    users:(("python",pid=18762,fd=60))     
NVIDIA A100-SXM4-40GB, 2520 MiB, 37922 MiB, 0 %
Loading EVO_1 model...
/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. D

In [53]:
# 13. Websocket-only test — pure Python version

import subprocess, os

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

ws_test_code = r'''
import asyncio
import websockets

async def main():
    url = "ws://127.0.0.1:9010"
    print("trying", url)
    async with websockets.connect(
        url,
        max_size=100_000_000,
        ping_interval=None,
        ping_timeout=None,
        close_timeout=30,
    ) as ws:
        print("CONNECTED OK")

asyncio.run(main())
'''

subprocess.run(
    [MAMBA, "run", "-n", "libero", "python", "-c", ws_test_code],
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    check=True,
)

CompletedProcess(args=['/content/micromamba/bin/micromamba', 'run', '-n', 'libero', 'python', '-c', '\nimport asyncio\nimport websockets\n\nasync def main():\n    url = "ws://127.0.0.1:9010"\n    print("trying", url)\n    async with websockets.connect(\n        url,\n        max_size=100_000_000,\n        ping_interval=None,\n        ping_timeout=None,\n        close_timeout=30,\n    ) as ws:\n        print("CONNECTED OK")\n\nasyncio.run(main())\n'], returncode=0)

In [61]:
# FP16 eval settings — 4 suites x 10 tasks x 10 episodes, resumable
import json, time
from pathlib import Path

TASK_SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
TASK_IDS = list(range(10))
EPISODES = list(range(10))
EXPECTED_EVAL_RUNS = len(TASK_SUITES) * len(TASK_IDS) * len(EPISODES)

MAX_STEPS_BY_SUITE = {
    'libero_spatial': 25,
    'libero_object': 25,
    'libero_goal': 25,
    'libero_10': 95,
}

TAG = 'fp16_baseline'
RESULTS = Path('/content/drive/MyDrive/cat_baseline/results')
EVAL_RESULTS = RESULTS / 'eval_4suites_10ep'
EVAL_RESULTS.mkdir(parents=True, exist_ok=True)

manifest = {
    'tag': TAG,
    'task_suites': TASK_SUITES,
    'task_ids': TASK_IDS,
    'episodes': EPISODES,
    'expected_eval_runs': EXPECTED_EVAL_RUNS,
    'max_steps_by_suite': MAX_STEPS_BY_SUITE,
    'success_rule': 'episode log contains checkmark Success only',
    'quant': 'none_fp16',
    'created_time': time.time(),
}
(EVAL_RESULTS / f'{TAG}_benchmark_manifest.json').write_text(json.dumps(manifest, indent=2))

print('TAG:', TAG)
print('EVAL_RESULTS:', EVAL_RESULTS)
print('TASK_SUITES:', TASK_SUITES)
print('EXPECTED_EVAL_RUNS:', EXPECTED_EVAL_RUNS)
print('MAX_STEPS_BY_SUITE:', MAX_STEPS_BY_SUITE)
print('RESUME_RULE: completed episodes have *.done.json and will be skipped on rerun.')


Total subprocess runs: 10


In [ ]:
# FP16 run loop — per-task and per-suite results, overall result, accuracy saved
import subprocess, os, time, json
from pathlib import Path

MAMBA = '/content/micromamba/bin/micromamba'
MAMBA_ROOT = '/content/micromamba-root'
LIBERO_EVAL = '/content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation'

def server_listening():
    r = subprocess.run("ss -ltnp | grep '9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0

def run_one(suite, task_id, ep):
    name = f'{TAG}_{suite}_task{task_id}_ep{ep}'
    log = EVAL_RESULTS / f'{name}.log'
    done = EVAL_RESULTS / f'{name}.done.json'
    if done.exists():
        try:
            prev = json.loads(done.read_text())
            result = 'SUCCESS' if bool(prev.get('success')) else 'FAIL'
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done ({result})', flush=True)
        except Exception as exc:
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done (unreadable: {exc})', flush=True)
        return
    if not server_listening():
        raise RuntimeError('Server not listening on 9010')
    env = {**os.environ, 'MAMBA_ROOT_PREFIX': MAMBA_ROOT, 'SINGLE_SUITE': suite,
           'SINGLE_TASK_ID': str(task_id), 'SINGLE_EP_INDEX': str(ep),
           'SINGLE_MAX_STEPS': str(MAX_STEPS_BY_SUITE[suite]), 'SINGLE_CKPT_NAME': name}
    with open(log, 'w') as f:
        p = subprocess.run([MAMBA, 'run', '-n', 'libero', 'python', '-u', 'libero_client_single_episode_runtime.py'],
                           cwd=LIBERO_EVAL, env=env, stdout=f, stderr=subprocess.STDOUT, text=True)
    text = log.read_text(errors='ignore')
    egl = ('EGL_NOT_INITIALIZED' in text or 'EGLGLContext.__del__' in text)
    crash = (p.returncode != 0) or ('Traceback' in text and not (p.returncode == 0 and egl))
    success = ('✅ Success' in text)
    rec = {'suite': suite, 'task_id': task_id, 'episode': ep, 'returncode': p.returncode,
           'success': success, 'task_fail': (not success and not crash), 'server_crash': crash,
           'egl_cleanup_warning': egl, 'log_path': str(log), 'time': time.time()}
    done.write_text(json.dumps(rec, indent=2))
    print(f'EPISODE_RESULT suite={suite} task={task_id:02d} ep={ep:02d} ' + ('SUCCESS' if success else 'FAIL') + f' crash={crash}', flush=True)
    if crash or not success:
        print('EPISODE_LOG_TAIL_BEGIN')
        print('
'.join(text.splitlines()[-25:]))
        print('EPISODE_LOG_TAIL_END')

overall_done = 0
overall_success = 0
run_suite_results = {}
run_task_results = {}
for s in TASK_SUITES:
    suite_done = 0
    suite_success = 0
    run_task_results[s] = {}
    for t in TASK_IDS:
        for e in EPISODES:
            run_one(s, t, e)
        task_records = []
        for e in EPISODES:
            p = EVAL_RESULTS / f'{TAG}_{s}_task{t}_ep{e}.done.json'
            if p.exists():
                try:
                    task_records.append(json.loads(p.read_text()))
                except Exception as exc:
                    print(f'TASK_RESULT_READ_WARN suite={s} task={t:02d} ep={e:02d} {exc}', flush=True)
        wins = sum(1 for r in task_records if bool(r.get('success')))
        n = len(task_records)
        suite_done += n
        suite_success += wins
        run_task_results[s][t] = {'success': wins, 'total': n, 'rate': wins / n if n else 0}
        print(f'TASK_RESULT suite={s} task={t:02d} success={wins}/{n} rate={(wins / n if n else 0):.3f}', flush=True)
    run_suite_results[s] = {'success': suite_success, 'total': suite_done, 'rate': suite_success / suite_done if suite_done else 0}
    overall_done += suite_done
    overall_success += suite_success
    print(f'SUITE_PROGRESS suite={s} success={suite_success}/{suite_done} rate={(suite_success / suite_done if suite_done else 0):.3f}', flush=True)

overall_rate = overall_success / overall_done if overall_done else 0
print(f'OVERALL_RESULT success={overall_success}/{overall_done} rate={overall_rate:.3f}', flush=True)
acc_record = {
    'tag': TAG,
    'quant': 'none_fp16',
    'overall': {'success': overall_success, 'total': overall_done, 'rate': overall_rate},
    'by_suite': run_suite_results,
    'by_task': {s: {str(t): v for t, v in tasks.items()} for s, tasks in run_task_results.items()},
}
ACC_PATH = EVAL_RESULTS / f'{TAG}_run_accuracy.json'
ACC_PATH.write_text(json.dumps(acc_record, indent=2))
print(f'ACCURACY_SAVED: {ACC_PATH}', flush=True)


\nRUN fp16_spatial_task0_10ep_libero_spatial_task0_ep0
DONE fp16_spatial_task0_10ep_libero_spatial_task0_ep0 returncode= 0 success= True crashed= True
\n--- tail ---
    EGL.eglDestroyContext(EGL_DISPLAY, self._context)\n  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/OpenGL/error.py", line 230, in glCheckError\n    raise self._errorClass(\nOpenGL.raw.EGL._errors.EGLError: 2026-05-13 11:24:30,767 [INFO] Failed to load library ( 'libGLU.so.0' ): libGLU.so.0: cannot open shared object file: No such file or directory\nEGLError(\n	err = EGL_NOT_INITIALIZED,\n	baseOperation = eglDestroyContext,\n	cArguments = (\n		<OpenGL._opaque.EGLDisplay_pointer object at 0x7d616f543ec0>,\n		<OpenGL._opaque.EGLContext_pointer object at 0x7d616f5432c0>,\n	),\n	result = 0\n)\nException ignored in: <function EGLGLContext.__del__ at 0x7d6182f75550>\nTraceback (most recent call last):\n  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/robosuite/renderers/context

In [ ]:
# 16. Combine per-episode summaries
import json
from pathlib import Path
import pandas as pd

RESULTS = Path("/content/drive/MyDrive/cat_baseline/results")
records = []
for p in sorted(RESULTS.glob(f"{TAG}_*.done.json")):
    records.append(json.loads(p.read_text()))

df = pd.DataFrame(records)
if len(df) == 0:
    print("No done summaries yet.")
else:
    display(df)
    agg = df.groupby("suite").agg(
        runs=("log_path", "count"),
        success=("success_detected", "sum"),
        crashed=("crashed_detected", "sum"),
    )
    agg["success_rate"] = agg["success"] / agg["runs"]
    display(agg)

    out_csv = RESULTS / f"{TAG}_combined.csv"
    out_json = RESULTS / f"{TAG}_combined.json"
    df.to_csv(out_csv, index=False)
    out_json.write_text(json.dumps(records, indent=2))
    print("Saved:", out_csv)
    print("Saved:", out_json)


,suite,task_id,episode_index,returncode,success_detected,crashed_detected,log_path,timestamp
0,libero_spatial,0,0,0,True,True,/content/drive/MyDrive/Evo-1-results/fp16_resu...,1.778671e+09


,runs,success,crashed,success_rate
suite,,,,
libero_spatial,1,1,1,1.0


Saved: /content/drive/MyDrive/Evo-1-results/fp16_resumable/fp16_resumable_debug_combined.csv
Saved: /content/drive/MyDrive/Evo-1-results/fp16_resumable/fp16_resumable_debug_combined.json


In [ ]:
# # 17. Stop server when finished
# !pkill -f Evo1_server.py || true
# !ps -ef | grep Evo1_server.py | grep -v grep || echo "SERVER STOPPED"


## Expand after debug

After `task0 × episode0` works, edit cell 14.

Spatial reference:

```python
TASK_SUITES = ["libero_spatial"]
TASK_IDS = list(range(10))
EPISODES = list(range(5))
TAG = "fp16_spatial_10tasks_5ep"
```

Full reference:

```python
TASK_SUITES = ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
TASK_IDS = list(range(10))
EPISODES = list(range(10))
TAG = "fp16_full_4suites_10ep"
```
